PROMPT ENGINEERING IN LANGSMITH

Import environment variables

In [28]:
import os
from dotenv import load_dotenv

_ = load_dotenv(dotenv_path=".env", override=True)
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

Pull in Prompt from Prompthub. Specify the dataset we'd like to run our experiment on.

In [29]:
from langsmith import Client

client = Client(api_key=LANGSMITH_API_KEY)
prompt = client.pull_prompt("essay_5yo_consice", include_model=True, secrets={"OPENAI_API_KEY": os.getenv("OPENAI_API_KEY")})
dataset_name = "essay-writer-5yo"

d:\repos\langgraph_learn\venv\Lib\site-packages\langchain_core\load\load.py:822: UserWarning: WARNING! extra_headers is not default parameter.
                extra_headers was transferred to model_kwargs.
                Please confirm that extra_headers is what you intended.
  loaded_obj = {k: _load(v) for k, v in obj.items()}


Setup AI Application

In [30]:
# Init web search tool
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

Let's now crate our application, same as in the tracing module. This time, our prompt is the one pulled from `PromptHub`

In [31]:
from langchain_openai import ChatOpenAI
from langsmith import traceable

llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))

@traceable
def search(question):
  web_response = tavily.search(query=question, max_results=3)
  return "\n".join([d["content"] for d in web_response["results"]])

@traceable
def explain(question, context):
  result = prompt.invoke({"question": question, "context": context}) # very easy to use this prompt and paste params which are intialized inside the prompt

  # If result.content is a list of blocks, extract the text string
  if isinstance(result.content, list):
    return result.content[0].get("text", "")
  return result.content

@traceable
def main(question):
  context = search(question)
  answer = explain(question, context)
  return answer


Define Evaluators

Custom Code Evaluator

We'll first define a custom code evaluator, which are useful to measure deterministic or close-ended metrics.

In [32]:
# check if our application produces outputs that are <= 200 words long
def conciseness(outputs: dict) -> bool:
  words = outputs["output"].split(" ")
  return len(words) <= 200

LLM-as-a-Judge Evaluator. For open-ended metrics, it's can be powerful to use an LLM to score the outputs. Let's use an LLM to check whether our application produces correct outputs. First, let's define a scoring schema for our LLM to adhere to in its response.

In [33]:
from pydantic import BaseModel, Field

# Define a scoring schema that our LLM must adhere to
class CorrectnessScore(BaseModel):
  """Correctness score of the answer when compared to the reference answer."""
  score: int = Field(description="The score of the correctness of the answer, from 0 to 1")

We'll define a function to dive an LLM our application's outputs, alongside the reference outputs stored in our dataset. The LLM will then be able to reference the "right" output to judge if our application's answer meets our accuracy standards.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
  # Safely extract reference output whether it's under 'messages' or 'output'
  if "output" in reference_outputs:
    ref_text = reference_outputs["output"]
  elif "messages" in reference_outputs and len(reference_outputs["messages"]) > 0:
    ref_text = reference_outputs["messages"][0].get("content", "")
  else:
    ref_text = reference_outputs.get("answer", "")

  prompt = """
  You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rules:
  
  <Rubrics>
    A correct answer:
    - Provides accurate information
    - Uses suitable analogies and examples
    - Contains no factual errors
    - Is logically consistent

    When scoring, you should penalize:
    - Factual errors
    - Incoherent analogies and examples
    - Logical inconsistencies
  </Rubrics>

  <Instructions>
    - Carefully read the input and output
    - Use the reference output to determine if the model output contains errors
    - Focus whether the model output uses accurate analogies and is logically consistent
  </Instructions>

  <Reminder>
    The analogies in the output do not need to match the reference output exactly. Focus on logical consistency.
  </Reminder>

  <input>
    {}
  </input>

  <output>
    {}
  </output>

  Use the reference outputs below to help you evaluate the correctness of the response:
  <reference_outputs>
    {}
  </reference_outputs>
  """.format(inputs["question"], outputs["output"], ref_text)
  
  structured_llm = ChatOpenAI(model_name="gpt-4o", temperature=0).with_structured_output(CorrectnessScore)
  generation = structured_llm.invoke([HumanMessage(content=prompt)])
  return generation.score == 1

Define Run function. We'll define a function to run our application on the example inputs of our dataset. This is function that will be called when we run our experiment.

In [35]:
# Define a function to run your application
def run(inputs: dict):
  return main(inputs["question"])

Run Experiment

We have all the necessary components, so let's run our experiment!

In [36]:
from langsmith import evaluate

evaluate(
  run,
  data = dataset_name,
  evaluators = [correctness, conciseness],
  experiment_prefix = "eli5-experiment"
)

View the evaluation results for experiment: 'eli5-experiment-8ba8738e' at:
https://smith.langchain.com/o/e2c23198-a55f-4194-8f70-301a89f1e8a8/datasets/6d849b52-3e72-4326-b02d-5047c7357071/compare?selectedSessions=03869404-382c-49f2-882c-f512645c456c




0it [00:00, ?it/s]

Error running evaluator <DynamicRunEvaluator correctness> on run 01a07116-2434-7351-88ec-e06f894b8121: KeyError('output')
Traceback (most recent call last):
  File "d:\repos\langgraph_learn\venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 1689, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
        run=run,
        example=example,
        evaluator_run_id=evaluator_run_id,
    )
  File "d:\repos\langgraph_learn\venv\Lib\site-packages\langsmith\evaluation\evaluator.py", line 370, in evaluate_run
    result = self.func(
        run,
        example,
        langsmith_extra={"run_id": evaluator_run_id, "metadata": metadata},
    )
  File "d:\repos\langgraph_learn\venv\Lib\site-packages\langsmith\run_helpers.py", line 781, in wrapper
    function_result = run_container["context"].run(
        func, *args, **kwargs
    )
  File "d:\repos\langgraph_learn\venv\Lib\site-packages\langsmith\evaluation\evaluator.py", line 791, in wrappe

<ExperimentResults eli5-experiment-8ba8738e>